# Project pipeline

   ## Schools
mtl_schools.csv -> this notebook -> schools_json_final.geojson -> site -> my map

## Recreational sites
project_recreationsites.geojson -> this notebook -> recreation_filtered.geojson -> site -> my map
!Important to note! recreationsites was already a geojson file, it only needed to be fintered



## Part 1. Load Schools and filter the columns

This cell imports my schools and recreation files, it reads it and keeps on the relevant columns. it also creates a point with its coordinates for schools. it then also reads in the geogjson for recreational facilities, which it then filters to only keep features relevant for preschool and elementary level students. 

In [1]:
import json
import pandas as pd
import geopandas as gpd

DATA_DIR = "C:/Users/PC/OneDrive/Celine Uni/GEOG 464/project/data/"

# ---------- Schools ----------
keep_columns = ["COORD_X_LL84_IMM", "COORD_Y_LL84_IMM", "ORDRE_ENS", "SITE_WEB_ORGNS",
                "NOM_IMM", "CD_POSTL_GDUNO_ORGNS", "NOM_MUNCP_GDUNO_ORGNS"]
schools = pd.read_csv(DATA_DIR + "mtl_schools.csv", usecols=keep_columns)
schools = schools.dropna(subset=["COORD_X_LL84_IMM", "COORD_Y_LL84_IMM"])   # a point needs both coordinates

# Island of Montréal only: postal codes H1-H5, H8, H9 (H7 = Laval)
fsa = schools["CD_POSTL_GDUNO_ORGNS"].str.strip().str.upper()
schools = schools[fsa.str.match(r"H[1-589]", na=False)]

# Types of school to keep
wanted_orders = ["Primaire", "Préscolaire - Primaire", "Préscolaire - Primaire - Secondaire",
                 "Primaire - Secondaire", "Préscolaire"]
schools = schools[schools["ORDRE_ENS"].isin(wanted_orders)]

gdf_school = gpd.GeoDataFrame(
    schools,
    geometry=gpd.points_from_xy(schools["COORD_X_LL84_IMM"], schools["COORD_Y_LL84_IMM"]),  # lon, lat
    crs="EPSG:4326",
)
print("schools:", len(gdf_school), gdf_school.total_bounds)   # expect roughly [-73.9 45.4 -73.4 45.7]
gdf_school.to_file(DATA_DIR + "schools_json_final.geojson", driver="GeoJSON")

# ---------- Recreation sites ----------
rec = gpd.read_file(DATA_DIR + "project_recreationsites.geojson")
rec = rec[["ID", "NOM", "TYPE", "ARROND", "geometry"]]
rec = rec[rec["TYPE"].isin(["Sportif", "Récréatif", "Aire de jeu - enfant"])]
print("recreation:", len(rec))
rec.to_file(DATA_DIR + "recreation_filtered.geojson", driver="GeoJSON")

print(schools.head())
print(rec.head())


problems = []

schools: 348 [-73.880687  45.419518 -73.479247  45.700858]
recreation: 2651
  CD_POSTL_GDUNO_ORGNS NOM_MUNCP_GDUNO_ORGNS                    NOM_IMM  \
0               H3M2X9              Montréal  François-de-Laval, annexe   
3               H4B1A8              Montréal  Mackay / Philip E. Layton   
4               H4B1A8              Montréal  Mackay / Philip E. Layton   
6               H1S1J2              Montréal             Général-Vanier   
8               H4H2L1              Montréal           Coquelicots, des   

                                SITE_WEB_ORGNS  COORD_X_LL84_IMM  \
0  https://francois-de-laval.cssdm.gouv.qc.ca/        -73.695445   
3               https://www.emsb.qc.ca/mackay/        -73.633305   
4                                          NaN        -73.633305   
6            https://generalvanier.cspi.qc.ca/        -73.590024   
8       https://descoquelicots.ecoleverdun.com        -73.586991   

   COORD_Y_LL84_IMM                            ORDRE_ENS  
0    

##  Check the files

this check file will varify if all the data i need is included and if there is any missing data within my file.

In [2]:


problems = []

# Schools file
with open(DATA_DIR + "schools_json_final.geojson", encoding="utf-8") as f:
    back = json.load(f)

feats = back.get("features", [])
if back.get("type") != "FeatureCollection":
    problems.append("The schools file is not a FeatureCollection.")
if len(feats) != len(gdf_school):
    problems.append(f"The schools file has {len(feats)} features but the clean table has {len(gdf_school)} rows.")
lons = [ft["geometry"]["coordinates"][0] for ft in feats]
lats = [ft["geometry"]["coordinates"][1] for ft in feats]
if feats and not all(-74.1 < x < -73.4 for x in lons):
    problems.append("Some school longitudes are outside Montréal. Are longitude and latitude swapped?")
if feats and not all(45.3 < y < 45.8 for y in lats):                                                    #checnking if long and lat are changed?
    problems.append("Some school latitudes are outside Montréal. Are longitude and latitude swapped?")
if feats and not all({"NOM_IMM", "ORDRE_ENS", "CD_POSTL_GDUNO_ORGNS"} <= set(ft["properties"]) for ft in feats):
    problems.append("Some schools are missing NOM_IMM, ORDRE_ENS or CD_POSTL_GDUNO_ORGNS in their properties.")  # checnking for missing values
if feats and not all(ft["properties"]["ORDRE_ENS"] in wanted_orders for ft in feats):
    problems.append("Some schools have a school type you meant to remove.")  #checking the school types

# Recreation file
with open(DATA_DIR + "recreation_filtered.geojson", encoding="utf-8") as f:
    back_r = json.load(f)

feats_r = back_r.get("features", [])
if back_r.get("type") != "FeatureCollection":
    problems.append("The recreation file is not a FeatureCollection.")
if len(feats_r) != len(rec):
    problems.append(f"The recreation file has {len(feats_r)} features but the clean table has {len(rec)} rows.")
minx, miny, maxx, maxy = gpd.read_file(DATA_DIR + "recreation_filtered.geojson").total_bounds
if not (-74.1 < minx and maxx < -73.4 and 45.3 < miny and maxy < 45.8):  #same as schools, checking the coordinates
    problems.append("Some recreation sites are outside Montréal. Wrong coordinate system, or swapped coordinates?")
if feats_r and not all({"ID", "NOM", "TYPE", "ARROND"} <= set(ft["properties"]) for ft in feats_r):
    problems.append("Some recreation sites are missing ID, NOM, TYPE or ARROND in their properties.")  #checking for missing values
if feats_r and not all(ft["properties"]["TYPE"] in ["Sportif", "Récréatif", "Aire de jeu - enfant"] for ft in feats_r):
    problems.append("Some recreation sites have a TYPE you meant to remove.")  # checking the file to have the right data

print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))


All checks passed
